# EIM — Kaggle T4 × 2

This notebook installs the EIM app without replacing Kaggle's PyTorch build, runs offline checks, then launches the UI. Set **Internet = On** and **Accelerator = GPU T4 x2** in Kaggle Notebook settings. The temporary Gradio share link is public while active: do not use this session for private data or secrets.

In [ ]:
from pathlib import Path
import glob, os, shutil, subprocess, sys, zipfile

# Locate the uploaded project ZIP among Kaggle input datasets.
archives = glob.glob("/kaggle/input/**/*.zip", recursive=True)
if not archives:
    raise FileNotFoundError("No ZIP found under /kaggle/input. Add the EIM project ZIP as a Kaggle Dataset.")
# Prefer the original archive name if present; otherwise use the first attached ZIP.
archive = next((p for p in archives if Path(p).name == "Uncle-Card.zip"), archives[0])
project = Path("/kaggle/working/EIM")
if project.exists():
    shutil.rmtree(project)
project.mkdir(parents=True)
with zipfile.ZipFile(archive) as zf:
    # Guard against ZIP path traversal; only extract entries that remain under project.
    root = project.resolve()
    for info in zf.infolist():
        target = (project / info.filename).resolve()
        if target != root and root not in target.parents:
            print("Skipped unsafe archive path:", info.filename)
            continue
        if not info.is_dir():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(info.filename))
# Never keep the suspicious hidden config artifact from the supplied archive.
(project / ".exmu-cfg1.data").unlink(missing_ok=True)
print("Project:", project)
print("Archive:", archive)
print("Files:", len([p for p in project.rglob('*') if p.is_file()]))


In [ ]:
# Install app dependencies but retain Kaggle's existing CUDA-enabled torch.
%pip install -q -r /kaggle/working/EIM/requirements-kaggle.txt


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.chdir("/kaggle/working/EIM")

# Kaggle T4 profile. `device_map=auto` may spread model layers over both visible GPUs.
os.environ["EIM_BACKEND"] = "local"
os.environ["EIM_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["EIM_CHAT_MODEL"] = os.environ["EIM_MODEL"]
os.environ["EIM_TORCH_DTYPE"] = "fp16"
os.environ["EIM_PROFILE"] = "balanced"
os.environ["EIM_ALLOW_NETWORK"] = "0"
os.environ["EIM_ALLOW_RISKY_CODE"] = "0"
os.environ["EIM_ALLOW_OUTSIDE_WORKSPACE"] = "0"
os.environ["EIM_GRADIO_SHARE"] = "1"  # Temporary public URL; keep secrets out of this session.
os.environ.setdefault("EIM_CONTEXT_TOKENS", "4096")
os.environ.setdefault("EIM_MAX_NEW_TOKENS", "768")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i), f"VRAM={torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Set the Kaggle accelerator to GPU T4 x2 and restart the session.")


## Offline validation

These checks do not download model weights and do not measure live model quality. The EIM+ extended self-test is intentionally not run here because its subprocess/mutation suite can take a long time in notebook environments.

In [ ]:
import subprocess, sys
checks = [
    [sys.executable, "-m", "compileall", "-q", "."],
    [sys.executable, "app.py", "--selftest"],
    [sys.executable, "chat_app.py", "--selftest"],
    [sys.executable, "eval_eim.py", "--selftest"],
    [sys.executable, "test_correction_memory.py"],
    [sys.executable, "test_eval_benchmarks.py"],
    [sys.executable, "test_web_search.py"],
    [sys.executable, "test_deployment.py"],
    [sys.executable, "test_terminal_verify.py"],
]
failed=[]
for cmd in checks:
    print("\n$", " ".join(cmd), flush=True)
    try:
        result=subprocess.run(cmd, text=True, timeout=120, capture_output=True)
        print(result.stdout[-12000:])
        if result.stderr:
            print("STDERR:", result.stderr[-4000:])
        print("EXIT:", result.returncode)
        if result.returncode: failed.append((cmd, result.returncode))
    except subprocess.TimeoutExpired:
        print("TIMEOUT after 120 seconds")
        failed.append((cmd, "timeout"))
if failed:
    raise RuntimeError(f"Validation failures: {failed}")
print("\nAll selected offline checks passed.")


## Launch EIM

The first real response may download model weights and take several minutes. The model runs only when a request arrives. Keep this notebook session active while using the temporary Gradio URL.

In [ ]:
# This cell stays active while the Gradio app runs. Stop the cell to shut down the app.
subprocess.run([sys.executable, "chat_app.py"], check=True, cwd="/kaggle/working/EIM", env=os.environ.copy())
